# HyperNetworkCombiner: Conditional Feature Processing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ludwig-ai/ludwig/blob/main/examples/hypernetwork/hypernetwork.ipynb)

> **Note:** This notebook requires **Ludwig >= 0.14** (PR #4092). The `hypernetwork`
> combiner type is not available in earlier releases. Install with:
> `pip install "ludwig>=0.14"`

This notebook demonstrates the `HyperNetworkCombiner`, which lets one feature
(**the conditioning feature**) generate the weights of the layers that process all
other features — rather than simply concatenating everyone together.

Based on **HyperFusion** ([arXiv 2403.13319](https://arxiv.org/abs/2403.13319), 2024).

**What we cover:**

1. Why concatenation is not always enough
2. Generating a synthetic multi-modal sensor dataset
3. Baseline: concat combiner
4. HyperNetworkCombiner
5. Comparing results and understanding why hypernetwork wins

In [ ]:
!pip install "ludwig>=0.14" --quiet

## The problem: context-dependent features

Imagine a network of industrial sensors. Each sensor reports three readings
(`sensor_a`, `sensor_b`, `sensor_c`), but every sensor belongs to one of three
measurement types: **temperature**, **pressure**, or **humidity**.

The catch: **the same numerical reading means something completely different**
depending on the sensor type.

- For a **temperature** sensor, `sensor_a = 3.0` is an anomaly (overheating).
- For a **pressure** sensor, `sensor_a = 3.0` is perfectly normal.
- For a **humidity** sensor, the anomaly rule involves the *sum* of all three readings.

A concat combiner encodes all four features independently and then stitches them
together. The network has to learn — **after** the concatenation — to undo the mixing
and apply type-specific logic. This is hard because the critical signal (`sensor_type`)
is buried in a shared representation alongside the numerical readings.

The `hypernetwork` combiner solves this directly:

```
sensor_type  ──► HyperNetwork ──► generates weights W, b
                                       │
sensor_a ────────────────────► FC(W, b) ──►
sensor_b ────────────────────► FC(W, b) ──►  combined repr.
sensor_c ────────────────────► FC(W, b) ──►
```

`sensor_type` does not contribute a feature vector — it **rewrites the entire
transformation** applied to the numerical sensors.

## Dataset

We generate a synthetic dataset with three sensor types. Each type has its own
normal operating range and its own anomaly rule, making the sensor type an
essential piece of context for correct classification.

In [ ]:
import numpy as np
import pandas as pd

RNG = np.random.default_rng(42)

N_PER_TYPE = 600
SENSOR_TYPES = ["temperature", "pressure", "humidity"]


def make_samples(sensor_type: str, n: int, rng: np.random.Generator) -> pd.DataFrame:
    """Generate n samples for a single sensor type with type-specific anomaly rules."""
    if sensor_type == "temperature":
        # Normal: readings near 0; anomaly: sensor_a > 2.5 (overheating)
        sensor_a = rng.normal(0.0, 1.0, n)
        sensor_b = rng.normal(0.0, 1.0, n)
        sensor_c = rng.normal(0.0, 1.0, n)
        anomaly = (sensor_a > 2.5).astype(int)
    elif sensor_type == "pressure":
        # Normal: readings near 1; anomaly: sensor_b drops below -0.5 (leak)
        sensor_a = rng.normal(1.0, 0.8, n)
        sensor_b = rng.normal(1.0, 0.8, n)
        sensor_c = rng.normal(1.0, 0.8, n)
        anomaly = (sensor_b < -0.5).astype(int)
    else:  # humidity
        # Normal: readings near -1; anomaly: combined level exceeds threshold
        sensor_a = rng.normal(-1.0, 0.9, n)
        sensor_b = rng.normal(-1.0, 0.9, n)
        sensor_c = rng.normal(-1.0, 0.9, n)
        anomaly = ((sensor_a + sensor_b + sensor_c) > 0).astype(int)

    return pd.DataFrame(
        {
            "sensor_a": sensor_a,
            "sensor_b": sensor_b,
            "sensor_c": sensor_c,
            "sensor_type": sensor_type,
            "anomaly": anomaly,
        }
    )


frames = [make_samples(t, N_PER_TYPE, RNG) for t in SENSOR_TYPES]
df = pd.concat(frames, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

# Train / validation / test split  (70 / 15 / 15)
n = len(df)
split = np.full(n, 2, dtype=int)  # default: test
idx = np.arange(n)
RNG.shuffle(idx)
split[idx[: int(0.70 * n)]] = 0
split[idx[int(0.70 * n) : int(0.85 * n)]] = 1
df["split"] = split

print(f"Total rows: {n}")
print(f"Overall anomaly rate: {df['anomaly'].mean():.1%}")
print()
print("Anomaly rate by sensor type:")
print(df.groupby("sensor_type")["anomaly"].mean().rename("anomaly_rate"))
print()
df.head(10)

Notice that the three sensor types have **different anomaly rates** and, more
importantly, the anomaly rules are structurally different. The same value of
`sensor_a = 3.0` triggers an anomaly for `temperature` but not for `pressure`.
A model that treats all features symmetrically will struggle with this.

## Baseline: concat combiner

The default Ludwig combiner concatenates all encoder outputs and passes them
through fully-connected layers. `sensor_type` is just another input — its
embedding is concatenated alongside the numerical sensor values.

In [ ]:
import yaml
from ludwig.api import LudwigModel

config_concat_str = """
model_type: ecd

input_features:
  - name: sensor_a
    type: number
    preprocessing:
      normalization: zscore
  - name: sensor_b
    type: number
    preprocessing:
      normalization: zscore
  - name: sensor_c
    type: number
    preprocessing:
      normalization: zscore
  - name: sensor_type
    type: category

output_features:
  - name: anomaly
    type: binary

combiner:
  type: concat
  fc_layers:
    - output_size: 128
    - output_size: 64

trainer:
  epochs: 30
  learning_rate: 0.001
"""

config_concat = yaml.safe_load(config_concat_str)

model_concat = LudwigModel(config_concat, logging_level=30)
model_concat.train(dataset=df)

test_df = df[df["split"] == 2].copy()
preds_concat, _ = model_concat.predict(dataset=test_df)

acc_concat = (preds_concat["anomaly_predictions"].values == test_df["anomaly"].values).mean()
print(f"Concat combiner — test accuracy: {acc_concat:.4f}")

## HyperNetworkCombiner

Now we switch to `type: hypernetwork`. The combiner reads `sensor_type` (the last
feature listed in `input_features`) through a hyper-network and uses the output to
generate the weight matrix and bias of the layer that processes `sensor_a`,
`sensor_b`, and `sensor_c`.

Key parameters:

| Parameter | Role |
|---|---|
| `hidden_size` | Size of the main processing layer |
| `hyper_hidden_size` | Hidden size of the hyper-network itself |
| `output_size` | Dimension of the combined representation passed to decoders |

In [ ]:
config_hypernetwork_str = """
model_type: ecd

input_features:
  - name: sensor_a
    type: number
    preprocessing:
      normalization: zscore
  - name: sensor_b
    type: number
    preprocessing:
      normalization: zscore
  - name: sensor_c
    type: number
    preprocessing:
      normalization: zscore
  - name: sensor_type
    type: category

output_features:
  - name: anomaly
    type: binary

combiner:
  type: hypernetwork
  hidden_size: 128
  hyper_hidden_size: 64
  output_size: 128

trainer:
  epochs: 30
  learning_rate: 0.001
"""

config_hypernetwork = yaml.safe_load(config_hypernetwork_str)

model_hypernetwork = LudwigModel(config_hypernetwork, logging_level=30)
model_hypernetwork.train(dataset=df)

preds_hyper, _ = model_hypernetwork.predict(dataset=test_df)

acc_hyper = (preds_hyper["anomaly_predictions"].values == test_df["anomaly"].values).mean()
print(f"HyperNetworkCombiner — test accuracy: {acc_hyper:.4f}")

## Comparison

In [ ]:
import matplotlib.pyplot as plt

# Per-type breakdown
rows = []
for stype in SENSOR_TYPES:
    mask = test_df["sensor_type"] == stype
    true = test_df.loc[mask, "anomaly"].values

    acc_c = (preds_concat.loc[mask, "anomaly_predictions"].values == true).mean()
    acc_h = (preds_hyper.loc[mask, "anomaly_predictions"].values == true).mean()
    rows.append({"sensor_type": stype, "concat": round(acc_c, 4), "hypernetwork": round(acc_h, 4)})

rows.append({"sensor_type": "OVERALL", "concat": round(acc_concat, 4), "hypernetwork": round(acc_hyper, 4)})

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(rows))
width = 0.35
ax.bar(x - width / 2, results_df["concat"], width, label="Concat", color="steelblue")
ax.bar(x + width / 2, results_df["hypernetwork"], width, label="HyperNetwork", color="darkorange")
ax.set_xticks(x)
ax.set_xticklabels(results_df["sensor_type"])
ax.set_ylabel("Test accuracy")
ax.set_ylim(0.5, 1.0)
ax.set_title("Sensor anomaly detection: concat vs HyperNetworkCombiner")
ax.legend()
plt.tight_layout()
plt.show()

## Why hypernetwork wins

### The problem with concatenation

When we use `concat`, the model receives a vector like:

```
[enc(sensor_a), enc(sensor_b), enc(sensor_c), enc(sensor_type)]
```

The fully-connected layers after the concat have to learn — from scratch — that
`enc(sensor_type)` should *gate* the interpretation of the numerical sensors.
Effectively the network must implement a conditional logic as a series of
multiplications and additions over the entire concatenated vector. This is
theoretically possible but inefficient: many capacity-bearing parameters in the
FC layers end up implementing the routing rather than the actual anomaly detection.

### What hypernetwork does instead

The `HyperNetworkCombiner` separates the two roles explicitly:

1. **Hyper-network** — a small MLP that reads `sensor_type` and emits a vector
   of weights `W` and biases `b`.
2. **Main network** — a linear layer `FC(W, b)` applied to the concatenation of
   `[sensor_a, sensor_b, sensor_c]` using the *dynamically generated* `W` and `b`.

Because `W` and `b` are different for each sensor type, the transformation
applied to the numerical sensors is literally different per context. For
`temperature`, the generated `W` learns to make `sensor_a` highly predictive;
for `pressure`, the generated `W` shifts attention to `sensor_b`; for
`humidity`, it learns to combine all three.

### When to use it

- The conditioning feature is a **type, class, mode, or context** that changes
  the semantics of other features qualitatively.
- You have enough samples per conditioning category (roughly 200+ per class) to
  learn meaningful per-context transformations.
- The target signal requires different logic for different contexts, not just
  different magnitudes.

### When to stick with concat

- All features contribute on equal footing with no hierarchical conditioning.
- The dataset is very small — the hyper-network adds parameters.
- The extra complexity is not warranted (always start simple).